# DeepWings live website -- single-image query

Notebooks 21/26 work with DeepWings' **published dataset** (test/test_masks
downloaded from their GitHub) -- this notebook instead queries their **live
web model** directly (https://deepwings.ipb.pt/), since it has no
documented API, to get a genuine "their model on our data" data point,
complementing notebook 26's "our model on their data".

**No public API exists** (confirmed: no "Use via API" link, and the page
isn't even Gradio despite looking like it -- it's a custom Dropzone.js +
Socket.IO app). The request flow below was reverse-engineered from the
page's own JavaScript:

1. `POST /` (`multipart/form-data`): upload the image (`file` field) plus a
   random per-session `Tab` id (mirrors `sessionStorage.tabID` in their JS).
2. `POST /Processing` (`Tab` id): kicks off server-side processing, returns
   a loading page.
3. That loading page polls a Socket.IO connection every ~1s
   (`emit('my event', {data: tab_id})`) until the server emits `'recv6'`,
   signalling the result is ready -- ~8s for one image in testing. Their
   socket.io-client is v2.4.0 (older Engine.IO v3 protocol), hence pinning
   `python-socketio`/`python-engineio` to compatible major versions below.
4. `POST /info` (`Tab` id): returns the results page, with both the
   classification and landmark tables embedded as a JSON blob
   (`JSON.parse('...')`) that a client-side Tabulator.js table renders --
   there's no server-rendered `<table>` HTML to scrape, just this JSON.

**Scope note:** this notebook is for occasional single/few-image checks.
Scaling to the whole test set should wait for the go-ahead from DeepWings'
own maintainers -- there's no documented rate limit or bulk-access route,
and the UI itself caps manual uploads at 40 images, suggesting this small
academic tool wasn't built for large automated runs.

## Setup

In [1]:
import json
import random
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import socketio

from wings.config import PROCESSED_DATA_DIR

DEEPWINGS_BASE = "https://deepwings.ipb.pt"

2026-09-29 15:50:53.552 | INFO     | wings.config:<module>:44 - PROJ_ROOT path is: C:\Users\X\projects\bees


2026-09-29 15:50:53.572 | INFO     | wings.config:<module>:68 - torch.cuda.get_device_name()='NVIDIA RTX A3000 12GB Laptop GPU'


## Query function

In [2]:
def query_deepwings_website(image_path: Path, timeout_s: float = 90) -> dict:
    """Uploads one image to DeepWings' live website and returns its parsed
    classification + landmark results (see this notebook's intro for the
    reverse-engineered request flow). Raises TimeoutError if the Socket.IO
    'recv6' completion signal isn't received within timeout_s.

    Returns a dict with 'classification' (their subspecies/lineage output),
    'coords_raw' ((19,2) array, exactly their table's x1,y1..x19,y19, their
    own pixel scale/orientation -- see notebook intro on why this isn't
    directly comparable to our own pixel coordinates without a per-image
    similarity fit), and 'elapsed_s' (server processing time).
    """
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (research comparison, single-image query)"})
    tab_id = str(random.random())

    session.get(DEEPWINGS_BASE + "/")

    with open(image_path, "rb") as f:
        files = {"file": (image_path.name, f, "image/png")}
        session.post(DEEPWINGS_BASE + "/", files=files, data={"Tab": tab_id})

    session.post(DEEPWINGS_BASE + "/Processing", data={"Tab": tab_id})

    sio = socketio.Client(logger=False, engineio_logger=False)
    done = {"flag": False}

    @sio.on("recv6")
    def _on_recv6(data=None):
        done["flag"] = True

    cookies_header = "; ".join(f"{k}={v}" for k, v in session.cookies.get_dict().items())
    sio.connect(DEEPWINGS_BASE, headers={"Cookie": cookies_header} if cookies_header else {}, transports=["polling"])

    start = time.time()
    while not done["flag"] and time.time() - start < timeout_s:
        sio.emit("my event", {"data": tab_id})
        sio.sleep(1)
    sio.disconnect()

    if not done["flag"]:
        raise TimeoutError(f"DeepWings did not signal completion within {timeout_s}s for {image_path.name}")

    r = session.post(DEEPWINGS_BASE + "/info", data={"Tab": tab_id})
    m = re.search(r"JSON\.parse\('(.*?)'\)\];", r.text, re.DOTALL)
    if not m:
        raise ValueError(f"Could not find results JSON in /info response for {image_path.name}")
    classification_rows, landmark_rows = json.loads(m.group(1))

    row = landmark_rows[0]
    coords = np.array([[row[f"x{i}"], row[f"y{i}"]] for i in range(1, 20)], dtype=np.float64)
    return {"classification": classification_rows[0], "coords_raw": coords, "elapsed_s": time.time() - start}

## Demo: query our calibration image

`PL-0002-000188-R.dw.png` -- the same image used throughout this whole
comparison effort. Expected result (from manual + earlier scripted runs):
`A.m.mellifera` at 99.6344%, landmarks starting `(78,199), (71,211), ...`.

In [3]:
IMAGE_PATH = PROCESSED_DATA_DIR / "cropped" / "PL-wing-images" / "PL-0002-000188-R.dw.png"

result = query_deepwings_website(IMAGE_PATH)
print(f"Processed in {result['elapsed_s']:.1f}s")
print(f"Classification: {result['classification']['subspecies']} ({result['classification']['s_confidence']:.2f}%)")
print(f"Raw landmark coords (their pixel space, their column order):\n{result['coords_raw']}")

Processed in 9.1s
Classification: A.m.mellifera (99.63%)
Raw landmark coords (their pixel space, their column order):
[[ 78. 199.]
 [ 71. 211.]
 [ 98. 198.]
 [161. 163.]
 [141. 179.]
 [175. 214.]
 [170. 237.]
 [161. 252.]
 [182. 186.]
 [204. 157.]
 [195. 197.]
 [206. 169.]
 [227. 175.]
 [269. 178.]
 [254. 251.]
 [263. 207.]
 [284. 221.]
 [298. 220.]
 [370. 184.]]


## Comparing against our own ground truth

DeepWings' output table has *named* columns (x1,y1..x19,y19), so its
landmark numbering is a fixed convention -- not a per-image detection
order -- and shouldn't vary across images. The permutation below was
calibrated once on this same image via `handle_coordinates` (GPA +
Hungarian matching against `mean_coords`), cross-checked against a
"no reorder" baseline (disparity 0.8336) and a structured cyclic-shift
alternative (0.1316): this permutation reaches disparity 0.0006, only
possible if it's the true correspondence rather than a coincidental fit.

Scale + rotation + translation are still fit **per image** (proper,
no-reflection Kabsch, not GPA -- the order is already fixed): DeepWings
crops/zooms each upload differently (recovered scale ~1.85x on this image),
and it isn't yet confirmed whether the ~178 degree rotation found here is a
fixed pipeline constant or varies with how each photo happens to be framed
-- only one image has been checked so far. If a future run across several
images shows the same rotation every time, that could be frozen too.

In [4]:
DEEPWINGS_TO_OURS_PERM = [17, 16, 13, 15, 14, 12, 9, 11, 8, 10, 5, 6, 7, 3, 4, 2, 0, 1, 18]


def reorder_deepwings_raw(deepwings_raw: np.ndarray) -> np.ndarray:
    """deepwings_raw: (19,2) array in the order the website table gives
    (x1,y1..x19,y19). Returns the same raw values (untouched), reordered to
    our own canonical landmark order -- no GPA call, no coordinate changes."""
    deepwings_raw = np.asarray(deepwings_raw, dtype=np.float64)
    assert deepwings_raw.shape == (19, 2), f"expected (19,2), got {deepwings_raw.shape}"
    return deepwings_raw[DEEPWINGS_TO_OURS_PERM]


def kabsch_similarity(source: np.ndarray, target: np.ndarray):
    """Best-fit proper similarity transform (rotation + uniform scale +
    translation, no reflection) mapping source onto target in a
    least-squares sense. Returns (R, scale, t) such that
    target ~= scale * (source @ R.T) + t."""
    src_mean = source.mean(axis=0)
    tgt_mean = target.mean(axis=0)
    src_c = source - src_mean
    tgt_c = target - tgt_mean

    H = src_c.T @ tgt_c
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, d])
    R = Vt.T @ D @ U.T

    var_src = (src_c ** 2).sum()
    scale = np.trace(np.diag(S) @ D) / var_src
    t = tgt_mean - scale * (src_mean @ R.T)
    return R, scale, t


def deepwings_to_our_frame(deepwings_raw: np.ndarray, our_reference: np.ndarray):
    """Full pipeline for one image: fixed-order reorder (no GPA), then fit
    + apply the similarity transform onto our_reference (e.g. our ground
    truth, in our bottom-left/y-up pixel convention). Returns
    (transformed_points, R, scale, rotation_deg, t)."""
    ordered = reorder_deepwings_raw(deepwings_raw)
    R, scale, t = kabsch_similarity(ordered, our_reference)
    transformed = (ordered @ R.T) * scale + t
    rotation_deg = np.degrees(np.arctan2(R[1, 0], R[0, 0]))
    return transformed, R, scale, rotation_deg, t

In [5]:
from wings.metrics import wing_positional_precision

coords_df = pd.read_csv(PROCESSED_DATA_DIR / "cropped" / "PL-raw-coordinates.csv")
gt_row = coords_df[coords_df["file"] == IMAGE_PATH.name].iloc[0]
gt = gt_row.iloc[1:].to_numpy(dtype=np.float64).reshape(-1, 2)  # bottom-left/y-up, same convention as mean_coords

transformed, R, scale, rotation_deg, t = deepwings_to_our_frame(result["coords_raw"], gt)
per_lm_err = np.linalg.norm(transformed - gt, axis=1)
precision, _ = wing_positional_precision(gt, transformed)

print(f"Recovered transform: scale={scale:.4f} (our_px / their_px), rotation={rotation_deg:.1f} deg")
print(f"wing_positional_precision = {precision:.4f}")
print(f"pixel error vs our ground truth: mean={per_lm_err.mean():.3f}  median={np.median(per_lm_err):.3f}  max={per_lm_err.max():.3f}")

Recovered transform: scale=1.8452 (our_px / their_px), rotation=178.3 deg
wing_positional_precision = 0.9822
pixel error vs our ground truth: mean=2.117  median=1.276  max=10.187


## Full test set: 40-image batches through DeepWings' live model

**Authorized by DeepWings' own maintainers** (personal contacts) after the
bounded 120-image check below confirmed the batch mechanism works
correctly. This now runs the *entire* canonical test set (2172 images,
`WingsRawDataset(COUNTRIES, .../cropped).split(0.2, 0.1, seed=42)` -- the
same split `build_mask_datasets` uses for training, e.g. config 8d) through
their site, 40 images at a time (~55 batches, ~25-30 minutes total at the
~27s/batch observed on the 120-image run).

Mechanics unchanged from the bounded test: Dropzone's indexed `file[i]`
field names (see the fix note on `query_deepwings_batch` below), Socket.IO
`recv6` completion signal, `(0,0)`-coordinate rows treated as detection
failures and excluded from accuracy stats. **New for this longer run:**
results are cached to disk incrementally, after each batch, not just at the
end -- a run this long is worth being able to resume rather than restart
from scratch if a single batch times out or the connection drops.

In [6]:
from wings.dataset import WingsRawDataset
from wings.config import COUNTRIES, IMG_FOLDER_SUFX

BATCH_SIZE = 40

raw = WingsRawDataset(COUNTRIES, PROCESSED_DATA_DIR / "cropped")
_, _, test_subset = raw.split(val_percentage=0.2, test_percentage=0.1, seed=42)
test_indices = list(test_subset.indices)
print(f"Full canonical test set: {len(test_indices)} images")

sample_indices = test_indices  # the whole test set, not a subsample
sample_files = [raw.coords_df.loc[i, "file"] for i in sample_indices]
sample_gt = {
    raw.coords_df.loc[i, "file"]: raw.coords_df.loc[i, "orig_label"].numpy().reshape(-1, 2).astype(np.float64)
    for i in sample_indices
}
n_batches = -(-len(sample_files) // BATCH_SIZE)  # ceil division, last batch may be smaller
print(f"Using all {len(sample_files)} test-set images, in {n_batches} batches of up to {BATCH_SIZE}")


def resolve_image_path(filename: str) -> Path:
    country = filename.split("-", 1)[0]
    return PROCESSED_DATA_DIR / "cropped" / f"{country}{IMG_FOLDER_SUFX}" / filename

  0%|          | 0/21722 [00:00<?, ?it/s]

 78%|███████▊  | 17028/21722 [00:00<00:00, 168673.18it/s]

100%|██████████| 21722/21722 [00:00<00:00, 163523.06it/s]

Full canonical test set: 2172 images
Using all 2172 test-set images, in 55 batches of up to 40


In [7]:
def query_deepwings_batch(image_paths: list[Path], timeout_s: float = 300) -> dict:
    """Same flow as query_deepwings_website but for several images (up to
    the site's own advertised max of 40) in one request. FIXED: Dropzone's
    `_getParamName(n)` (checked directly in dropzone.js 5.9.3's source) uses
    `paramName + "[" + n + "]"` when `uploadMultiple: true`, i.e. field names
    `file[0]`, `file[1]`, ... -- NOT all files repeated under the literal
    key "file" (an earlier version of this function did that, and the
    server only picked up one file per batch as a result: verified against
    a real 40-image manual upload in the browser, which returned all 40).

    Returns {'coords_by_file': {filename: (19,2) array}, 'elapsed_s',
    'n_requested', 'n_returned'} (AVERAGE row excluded)."""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (research comparison, bounded batch test)"})
    tab_id = str(random.random())

    session.get(DEEPWINGS_BASE + "/")

    handles = [open(p, "rb") for p in image_paths]
    try:
        files_payload = [
            (f"file[{i}]", (p.name, fh, "image/png"))
            for i, (p, fh) in enumerate(zip(image_paths, handles))
        ]
        session.post(DEEPWINGS_BASE + "/", files=files_payload, data={"Tab": tab_id})
    finally:
        for fh in handles:
            fh.close()

    session.post(DEEPWINGS_BASE + "/Processing", data={"Tab": tab_id})

    sio = socketio.Client(logger=False, engineio_logger=False)
    done = {"flag": False}

    @sio.on("recv6")
    def _on_recv6(data=None):
        done["flag"] = True

    cookies_header = "; ".join(f"{k}={v}" for k, v in session.cookies.get_dict().items())
    sio.connect(DEEPWINGS_BASE, headers={"Cookie": cookies_header} if cookies_header else {}, transports=["polling"])

    start = time.time()
    while not done["flag"] and time.time() - start < timeout_s:
        sio.emit("my event", {"data": tab_id})
        sio.sleep(1)
    sio.disconnect()

    if not done["flag"]:
        raise TimeoutError(f"DeepWings did not signal completion within {timeout_s}s for a batch of {len(image_paths)}")

    r = session.post(DEEPWINGS_BASE + "/info", data={"Tab": tab_id})
    m = re.search(r"JSON\.parse\('(.*?)'\)\];", r.text, re.DOTALL)
    if not m:
        raise ValueError("Could not find results JSON in /info response")
    _classification_rows, landmark_rows = json.loads(m.group(1))

    results = {}
    for row in landmark_rows:
        fname = row["fileName"]
        if fname == "AVERAGE":
            continue
        results[fname] = np.array([[row[f"x{i}"], row[f"y{i}"]] for i in range(1, 20)], dtype=np.float64)

    return {
        "coords_by_file": results,
        "elapsed_s": time.time() - start,
        "n_requested": len(image_paths),
        "n_returned": len(results),
    }

In [8]:
from wings.config import REPORTS_DIR

CACHE_PATH = REPORTS_DIR / "deepwings_full_test_raw_coords.json"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)


def load_cache() -> dict:
    if not CACHE_PATH.exists():
        return {}
    with open(CACHE_PATH) as f:
        return {k: np.array(v, dtype=np.float64) for k, v in json.load(f).items()}


def save_cache(coords_dict: dict) -> None:
    with open(CACHE_PATH, "w") as f:
        json.dump({k: v.tolist() for k, v in coords_dict.items()}, f)


all_coords_raw = load_cache()
if all_coords_raw:
    print(f"Resuming: {len(all_coords_raw)}/{len(sample_files)} already cached from a previous run")

remaining_files = [f for f in sample_files if f not in all_coords_raw]
remaining_batches = [remaining_files[i:i + BATCH_SIZE] for i in range(0, len(remaining_files), BATCH_SIZE)]
print(f"{len(remaining_files)} images left to fetch, in {len(remaining_batches)} batches")

for batch_num, batch_files in enumerate(remaining_batches, start=1):
    batch_paths = [resolve_image_path(f) for f in batch_files]
    print(f"Uploading batch {batch_num}/{len(remaining_batches)} ({len(batch_paths)} images)...")
    try:
        batch_result = query_deepwings_batch(batch_paths)
        print(f"  done in {batch_result['elapsed_s']:.1f}s, got {batch_result['n_returned']}/{batch_result['n_requested']} results back")
        all_coords_raw.update(batch_result["coords_by_file"])
    except Exception as e:
        print(f"  BATCH FAILED: {e!r} -- will be retried on the next run (cache saved so far, nothing lost)")
    save_cache(all_coords_raw)  # incremental: survives a crash/interrupt partway through

print(f"\nTotal collected: {len(all_coords_raw)}/{len(sample_files)}")
missing = set(sample_files) - set(all_coords_raw)
if missing:
    print(f"Still missing ({len(missing)}): {sorted(missing)[:20]}{'...' if len(missing) > 20 else ''}")

2172 images left to fetch, in 55 batches
Uploading batch 1/55 (40 images)...


  done in 25.1s, got 40/40 results back
Uploading batch 2/55 (40 images)...


  done in 25.2s, got 40/40 results back
Uploading batch 3/55 (40 images)...


  done in 27.1s, got 40/40 results back
Uploading batch 4/55 (40 images)...


  done in 26.1s, got 40/40 results back
Uploading batch 5/55 (40 images)...


...

  done in 25.1s, got 40/40 results back
Uploading batch 52/55 (40 images)...


  done in 26.1s, got 40/40 results back
Uploading batch 53/55 (40 images)...


  done in 26.1s, got 40/40 results back
Uploading batch 54/55 (40 images)...


  done in 30.1s, got 40/40 results back
Uploading batch 55/55 (12 images)...


  done in 13.1s, got 12/12 results back

Total collected: 2172/2172


In [9]:
def is_detection_failure(raw_coords: np.ndarray) -> bool:
    """DeepWings appears to signal 'couldn't determine the right number of
    landmarks' by returning all-(0,0) coordinates for that image, rather
    than e.g. NaN or a shorter row -- a real landmark exactly at pixel
    (0,0) isn't plausible, so treat an all-zero result as a detection
    failure and exclude it from accuracy statistics (this is what the
    earlier try/except around the Kabsch fit was already doing implicitly
    -- an all-zero source has zero variance, so `scale` divides by zero and
    the fit raises -- this just makes that check explicit and intentional
    instead of relying on it as an incidental side effect)."""
    return np.all(raw_coords == 0)

n_failed = sum(1 for c in all_coords_raw.values() if is_detection_failure(c))
n_total = len(all_coords_raw)
print(f"Detection failures (model didn't find 19 usable points, all coords zero): {n_failed}/{n_total} ({n_failed / n_total:.1%})")
print(f"Successfully detected: {n_total - n_failed}/{n_total}")

per_image_precision = []
per_image_mean_err = []
unexpected_failures = []

for fname, raw_coords in all_coords_raw.items():
    if is_detection_failure(raw_coords):
        continue
    gt_coords = sample_gt[fname]
    try:
        transformed, R, scale, rotation_deg, t = deepwings_to_our_frame(raw_coords, gt_coords)
        precision, _ = wing_positional_precision(gt_coords, transformed)
        err = np.linalg.norm(transformed - gt_coords, axis=1)
        per_image_precision.append(precision)
        per_image_mean_err.append(err.mean())
    except Exception as e:
        unexpected_failures.append((fname, str(e)))

per_image_precision = np.array(per_image_precision)
per_image_mean_err = np.array(per_image_mean_err)

print(f"\nAccuracy statistics on the {len(per_image_precision)} successfully-detected images "
      f"({len(unexpected_failures)} unexpected fit errors among those, not detection failures):")
print(f"\nwing_positional_precision:")
print(f"  mean={per_image_precision.mean():.4f}  median={np.median(per_image_precision):.4f}  "
      f"min={per_image_precision.min():.4f}  max={per_image_precision.max():.4f}")
print(f"\nper-image mean pixel error (in our pixel space):")
print(f"  mean={per_image_mean_err.mean():.3f}px  median={np.median(per_image_mean_err):.3f}px  "
      f"min={per_image_mean_err.min():.3f}px  max={per_image_mean_err.max():.3f}px")
if unexpected_failures:
    print(f"\nUnexpected fit errors ({len(unexpected_failures)}):")
    for fname, err in unexpected_failures[:10]:
        print(f"  {fname}: {err}")

Detection failures (model didn't find 19 usable points, all coords zero): 68/2172 (3.1%)
Successfully detected: 2104/2172



Accuracy statistics on the 2104 successfully-detected images (0 unexpected fit errors among those, not detection failures):

wing_positional_precision:
  mean=0.9820  median=0.9830  min=0.7121  max=0.9880

per-image mean pixel error (in our pixel space):
  mean=2.400px  median=2.159px  min=1.542px  max=84.293px


## Summary

- `query_deepwings_website`/`query_deepwings_batch` get classification +
  raw landmarks from the live site (single image or up to ~40 at once) with
  no server-rendered table to scrape -- results come back as an embedded
  JSON blob. Batching needs Dropzone's *indexed* `file[0]`, `file[1]`, ...
  field names, not the literal key `file` repeated -- confirmed directly
  from dropzone.js 5.9.3's `_getParamName` source after an earlier version
  of this notebook got only 1/40 results per batch by getting this wrong.
- `DEEPWINGS_TO_OURS_PERM` is a one-time-calibrated, fixed landmark-order
  mapping (not re-derived via GPA per image) -- their column numbering is a
  named convention, not a per-image detection order.
- `deepwings_to_our_frame` fits (not searches) the per-image scale/rotation/
  translation needed to bring their raw coordinates into our own pixel
  space, so pixel errors are directly comparable to our own models' numbers
  (see notebooks 19/25/26 for those).
- `(0,0)` coordinates signal a per-image detection failure on their end
  (not a real landmark position) and are excluded from accuracy stats,
  counted separately as a failure rate.

**Full canonical test set (2172 images), authorized by DeepWings' own
maintainers after the bounded 120-image check below passed:**

- Detection failures (all-zero coordinates): 68/2172 (3.1%)
- On the 2104 successfully-detected images: `wing_positional_precision`
  mean=0.9820, median=0.9830 (min=0.7121, max=0.9880); pixel error (our
  pixel space) mean=2.400px, median=2.159px (min=1.542px, max=84.293px)

**In context of the rest of this project's cross-domain comparisons:**

| | precision |
|---|---|
| our model, our domain (notebook 25, in-domain) | ~0.99+ |
| our model, DeepWings' domain (notebook 26) | 0.91-0.95 |
| **DeepWings' live model, our domain (this notebook)** | **0.982** |
| DeepWings' own paper, their domain | 0.943 |

DeepWings' model does *better* on our test set than their own paper reports
for their own domain, and better than our model does on theirs. This reads
as a domain-difficulty effect, not a model-quality one: our collection
protocol produces cleaner, more consistent images than DeepWings' own
(messier field-collected photos, as established earlier in this project),
and that seems to help *any* model, not just ours -- consistent with our
own detection-failure rate on their domain (~9-10% outside the reliable
16-22 point range) being noticeably higher than theirs on ours (3.1%).

Raw per-image coordinates are cached in `reports/deepwings_full_test_raw_coords.json`
(and the earlier 120-image bounded check in `deepwings_120_raw_coords.json`)
so this analysis can be revisited without re-querying their server.